# 01 · Data Loading & SQL Pipeline
**Telco Customer Churn Prediction with SQL and Machine Learning**  
*Sections 1-5* · MSc Data Science project · Dr. Spoorthi H S

**In this notebook**
- Load the IBM Telco Customer Churn dataset (7,043 customers, 21 columns)
- Design a normalised SQLite schema: customers, services, contracts, billing
- Insert and verify all 7,043 rows in each table
- Four analytical SQL queries: filtering, multi-table JOINs, GROUP BY aggregation and HAVING
- Load the joined dataset back into pandas

> **How to use these notebooks.** The outputs below come from one complete run in Google Colab. Each part builds on objects created earlier (the SQL database, the prepared DataFrame, trained models), so the part notebooks are for reading and review. To reproduce the results, run [`00_full_pipeline_run_all.ipynb`](00_full_pipeline_run_all.ipynb) top to bottom.

[Data Quality & Preprocessing Study →](02_preprocessing_study.ipynb)

---

# TELCO CUSTOMER CHURN ANALYSIS
## Complete Data Mining Workflow with SQL, Advanced Preprocessing, and Machine Learning

**Objective:** Predict customer churn using an SQL-integrated data pipeline, feature engineering and three tuned machine-learning models (Logistic Regression, Random Forest, XGBoost).

**Result:** On a stratified 20 % hold-out test set (1,409 customers), all three models reach **F1 ≈ 0.62-0.63** and **ROC-AUC 0.84**, catching **74-79 % of churners**. Customers in their first six months churn at **54 %**, against **14 %** for customers with two or more years of tenure.

**Dataset:** IBM Telco Customer Churn (7,043 customers, 21 columns) - https://www.kaggle.com/blastchar/telco-customer-churn

## 1. ENVIRONMENT SETUP

In [34]:
# Install required packages
import subprocess
import sys

packages = ['xgboost', 'shap', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])


In [35]:
# Core imports
import numpy as np
import pandas as pd
from scipy import stats
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *
import xgboost as xgb
from imblearn.over_sampling import SMOTE

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
from datetime import datetime

print('All libraries imported successfully')


All libraries imported successfully


**Load the Telco Customer Churn dataset**

In [36]:
# Load the Telco Customer Churn dataset
import pandas as pd
df = pd.read_csv('/content/Telcochurn.csv')  # Update path if needed
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


**Explore Dataset Structure**

We will examine the columns, data types, and basic statistics to understand the dataset's structure.

In [37]:
# Display dataset columns and types
df.info()
# Show basic statistics
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
count,7043,7043,7043.000000,7043,7043,7043.000000,7043,7043,7043,7043,...,7043,7043,7043,7043,7043,7043,7043,7043.000000,7043,7043
unique,7043,2,NaN,2,2,NaN,2,3,3,3,...,3,3,3,3,3,2,4,NaN,6531,2
top,3186-AJIEK,Male,NaN,No,No,NaN,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,NaN,,No
freq,1,3555,NaN,3641,4933,NaN,6361,3390,3096,3498,...,3095,3473,2810,2785,3875,4171,2365,NaN,11,5174
mean,NaN,NaN,0.162147,NaN,NaN,32.371149,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.761692,NaN,NaN
std,NaN,NaN,0.368612,NaN,NaN,24.559481,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.090047,NaN,NaN
min,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.250000,NaN,NaN
25%,NaN,NaN,0.000000,NaN,NaN,9.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.500000,NaN,NaN
50%,NaN,NaN,0.000000,NaN,NaN,29.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.350000,NaN,NaN
75%,NaN,NaN,0.000000,NaN,NaN,55.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.850000,NaN,NaN


## 2. SQL DATABASE CREATION & SCHEMA DESIGN

In [38]:
import os
db_path = '/tmp/telco_churn.db'
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON')

# Create normalized schema
cursor.execute('''
    CREATE TABLE customers (
        customerID TEXT PRIMARY KEY,
        gender TEXT, SeniorCitizen INT, Partner TEXT,
        Dependents TEXT, tenure INT, Churn TEXT)
''')

cursor.execute('''
    CREATE TABLE services (
        serviceID INTEGER PRIMARY KEY,
        customerID TEXT UNIQUE,
        PhoneService TEXT, MultipleLines TEXT,
        InternetService TEXT, OnlineSecurity TEXT,
        OnlineBackup TEXT, DeviceProtection TEXT,
        TechSupport TEXT, StreamingTV TEXT,
        StreamingMovies TEXT,
        FOREIGN KEY(customerID) REFERENCES customers(customerID))
''')

cursor.execute('''
    CREATE TABLE contracts (
        contractID INTEGER PRIMARY KEY,
        customerID TEXT UNIQUE,
        Contract TEXT, PaperlessBilling TEXT,
        PaymentMethod TEXT,
        FOREIGN KEY(customerID) REFERENCES customers(customerID))
''')

cursor.execute('''
    CREATE TABLE billing (
        billingID INTEGER PRIMARY KEY,
        customerID TEXT UNIQUE,
        MonthlyCharges REAL, TotalCharges REAL,
        FOREIGN KEY(customerID) REFERENCES customers(customerID))
''')

conn.commit()
print('Database schema created successfully')


Database schema created successfully


## 3. DATA IMPORT INTO SQL TABLES

In [39]:
from google.colab import files
print('Upload Telcochurn.csv file...')
uploaded = files.upload()
csv_file = list(uploaded.keys())[0]
raw_data = pd.read_csv(csv_file)
print(f'Loaded: {raw_data.shape}')


Upload Telcochurn.csv file...


Saving Telcochurn.csv to Telcochurn (2).csv
Loaded: (7043, 21)


In [40]:
# Insert data into normalized tables
# Ensure TotalCharges is numeric, coercing errors to NaN
raw_data['TotalCharges'] = pd.to_numeric(raw_data['TotalCharges'], errors='coerce')

# Ensure SeniorCitizen is integer, coercing errors to NaN and then filling with 0
# This is crucial as the SQL schema expects an INT.
raw_data['SeniorCitizen'] = pd.to_numeric(raw_data['SeniorCitizen'], errors='coerce').fillna(0).astype(int)

# Group data for insertion into respective tables
customer_data = raw_data[['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'Churn']]
service_data = raw_data[['customerID', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']]
contract_data = raw_data[['customerID', 'Contract', 'PaperlessBilling', 'PaymentMethod']]
billing_data = raw_data[['customerID', 'MonthlyCharges', 'TotalCharges']]

tables_info = {
    'customers': customer_data,
    'services': service_data,
    'contracts': contract_data,
    'billing': billing_data
}

try:
    # Use a transaction for atomic insertion
    cursor.execute('BEGIN TRANSACTION')
    for table_name, df_to_insert in tables_info.items():
        df_to_insert.to_sql(table_name, conn, if_exists='append', index=False)
        print(f'Attempted to insert data into {table_name}.')
    conn.commit()
    print('Data insertion process completed and committed.')
except Exception as e:
    print(f'An error occurred during data insertion: {e}')
    conn.rollback()
    print('Transaction rolled back.')

# Verification step: Check row counts after insertion attempt
print('\nVerifying row counts in tables:')
try:
    for table_name in tables_info.keys():
        count_query = f"SELECT COUNT(*) FROM {table_name}"
        count = pd.read_sql_query(count_query, conn).iloc[0, 0]
        print(f"Table '{table_name}' has {count} rows.")
except Exception as e:
    print(f"Error verifying row counts: {e}")

Attempted to insert data into customers.
Attempted to insert data into services.
Attempted to insert data into contracts.
Attempted to insert data into billing.
Data insertion process completed and committed.

Verifying row counts in tables:
Table 'customers' has 7043 rows.
Table 'services' has 7043 rows.
Table 'contracts' has 7043 rows.
Table 'billing' has 7043 rows.


## 4. ADVANCED SQL QUERIES

In [41]:
# Query 1: SELECT subsets of data (with filtering)
query1 = '''
SELECT customerID, tenure, Churn, gender, SeniorCitizen
FROM customers
WHERE tenure > 12 AND Churn = 'No'
ORDER BY tenure DESC
LIMIT 15;
'''
print("\nQuery 1: SELECT subsets - Long-term customers who stayed")
print("="*70)
result1 = pd.read_sql_query(query1, conn)
print(result1)

# Query 2: JOIN tables
query2 = '''
SELECT c.customerID, c.tenure, c.Churn,
       b.MonthlyCharges, b.TotalCharges,
       s.InternetService, s.OnlineSecurity,
       ct.Contract, ct.PaymentMethod
FROM customers c
INNER JOIN billing b ON c.customerID = b.customerID
LEFT JOIN services s ON c.customerID = s.customerID
LEFT JOIN contracts ct ON c.customerID = ct.customerID
WHERE c.tenure > 6
ORDER BY c.tenure DESC
LIMIT 15;
'''
print("\nQuery 2: JOIN - Customer details with billing, services & contracts")
print("="*70)
result2 = pd.read_sql_query(query2, conn)
print(result2)

# Query 3: GROUP BY with Aggregation
query3 = '''
SELECT
    c.Churn,
    ct.Contract,
    s.InternetService,
    COUNT(*) as Total_Customers,
    ROUND(AVG(c.tenure), 1) as Avg_Tenure_Months,
    ROUND(AVG(b.MonthlyCharges), 2) as Avg_Monthly_Charges,
    ROUND(SUM(b.TotalCharges), 2) as Total_Revenue,
    ROUND(AVG(b.TotalCharges), 2) as Avg_Total_Charges
FROM customers c
INNER JOIN billing b ON c.customerID = b.customerID
LEFT JOIN contracts ct ON c.customerID = ct.customerID
LEFT JOIN services s ON c.customerID = s.customerID
GROUP BY c.Churn, ct.Contract, s.InternetService
ORDER BY Total_Customers DESC;
'''
print("\nQuery 3: GROUP BY & Aggregation - Churn analysis by contract & internet")
print("="*70)
result3 = pd.read_sql_query(query3, conn)
print(result3)

# Query 4: FILTER, GROUP, AGGREGATE (all requirements combined)
query4 = '''
SELECT
    c.SeniorCitizen,
    c.Partner,
    c.Dependents,
    ct.PaymentMethod,
    COUNT(*) as Customer_Count,
    ROUND(AVG(c.tenure), 1) as Avg_Tenure,
    ROUND(AVG(b.MonthlyCharges), 2) as Avg_Monthly_Charges,
    ROUND(SUM(CASE WHEN c.Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as Churn_Rate_Percent
FROM customers c
INNER JOIN billing b ON c.customerID = b.customerID
LEFT JOIN contracts ct ON c.customerID = ct.customerID
WHERE b.MonthlyCharges > 50  -- FILTER
GROUP BY c.SeniorCitizen, c.Partner, c.Dependents, ct.PaymentMethod  -- GROUP
HAVING COUNT(*) > 5  -- Additional filter on groups
ORDER BY Churn_Rate_Percent DESC;  -- AGGREGATE (ORDER BY uses aggregated column)
'''
print("\nQuery 4: FILTER, GROUP, AGGREGATE - Demographics & payment analysis")
print("="*70)
result4 = pd.read_sql_query(query4, conn)
print(result4)

# Summary of requirements met:
print("\n" + "="*70)
print("ASSESSMENT REQUIREMENTS CHECKLIST:")
print("="*70)
print("✓ Query 1: SELECT subsets of data (with WHERE clause)")
print("✓ Query 2: JOIN tables (INNER JOIN + LEFT JOIN on 4 tables)")
print("✓ Query 3: GROUP BY with Aggregations (COUNT, AVG, SUM, ROUND)")
print("✓ Query 4: FILTER (WHERE), GROUP (GROUP BY), AGGREGATE (COUNT, AVG, SUM)")
print("✓ All queries use ORDER BY for sorting")
print("✓ Query 4 includes HAVING clause for group filtering")


Query 1: SELECT subsets - Long-term customers who stayed
    customerID  tenure Churn  gender  SeniorCitizen
0   5248-YGIJN      72    No    Male              0
1   6234-RAAPL      72    No  Female              0
2   5954-BDFSG      72    No  Female              0
3   0526-SXDJP      72    No    Male              0
4   9848-JQJTX      72    No    Male              0
5   6728-DKUCO      72    No  Female              0
6   2848-YXSMW      72    No    Male              0
7   6734-PSBAW      72    No    Male              0
8   3146-MSEGF      72    No  Female              1
9   5997-OPVFA      72    No    Male              0
10  9680-NIAUV      72    No  Female              0
11  7255-SSFBC      72    No    Male              0
12  6348-SNFUS      72    No    Male              0
13  9286-BHDQG      72    No    Male              0
14  7083-MIOPC      72    No  Female              0

Query 2: JOIN - Customer details with billing, services & contracts
    customerID  tenure Churn  MonthlyChar

## 5. LOAD DATA FROM SQL TO PYTHON

In [42]:
# Load complete dataset from SQL
query = '''
SELECT c.customerID, c.gender, c.SeniorCitizen, c.Partner,
       c.Dependents, c.tenure, c.Churn,
       b.MonthlyCharges, b.TotalCharges,
       ct.Contract, ct.PaperlessBilling, ct.PaymentMethod,
       s.PhoneService, s.MultipleLines, s.InternetService,
       s.OnlineSecurity, s.OnlineBackup, s.DeviceProtection,
       s.TechSupport, s.StreamingTV, s.StreamingMovies
FROM customers c
INNER JOIN billing b ON c.customerID = b.customerID
LEFT JOIN contracts ct ON c.customerID = ct.customerID
LEFT JOIN services s ON c.customerID = s.customerID
'''
df = pd.read_sql_query(query, conn)
print(f'Data Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nFirst 5 rows:')
print(df.head())


Data Shape: (7043, 21)

Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'Churn', 'MonthlyCharges', 'TotalCharges', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

First 5 rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure Churn  \
0  7590-VHVEG  Female              0     Yes         No       1    No   
1  5575-GNVDE    Male              0      No         No      34    No   
2  3668-QPYBK    Male              0      No         No       2   Yes   
3  7795-CFOCW    Male              0      No         No      45    No   
4  9237-HQITU  Female              0      No         No       2   Yes   

   MonthlyCharges  TotalCharges        Contract  ...  \
0           29.85         29.85  Month-to-month  ...   
1           56.95       1889.50        One year  ...   
2           53.85    

---

[Next: Data Quality & Preprocessing Study →](02_preprocessing_study.ipynb)